In [19]:
import pandas as pd
import re
import emoji

In [20]:
# Cargar las 3 variantes de entrenamiento
df_mx_train = pd.read_csv('../data/irosva.mx.training.csv')
df_es_train = pd.read_csv('../data/irosva.es.training.csv')
df_cu_train = pd.read_csv('../data/irosva.cu.training.csv')

# Identificar variante dialectal de cada registro
df_mx_train['VARIANTE'] = 'mx'
df_es_train['VARIANTE'] = 'es'
df_cu_train['VARIANTE'] = 'cu'

# Unir y eliminar duplicados encontrados en exploración
df_train = pd.concat([df_mx_train, df_es_train, df_cu_train], ignore_index=True)
antes = len(df_train)
df_train = df_train.drop_duplicates(subset='MESSAGE').reset_index(drop=True)
print(f"Train: {antes} → {len(df_train)} registros ({antes - len(df_train)} duplicados eliminados)")

Train: 7200 → 7197 registros (3 duplicados eliminados)


In [21]:
# Cargar archivos de test y combinar con etiquetas reales
def cargar_test(variante, nombre_variante):
    df_texto = pd.read_csv(f'../data/irosva.{variante} - irosva.{variante}.test.csv')
    df_truth = pd.read_csv(f'../data/irosva.{variante}.test.truth.csv')
    # El archivo de texto tiene IS_IRONIC = "?" — se reemplaza con etiquetas reales
    df_texto = df_texto.drop(columns=['IS_IRONIC'])
    df = df_texto.merge(df_truth[['ID', 'IS_IRONIC']], on='ID')
    df['VARIANTE'] = nombre_variante
    return df

df_mx_test = cargar_test('mx', 'mx')
df_es_test = cargar_test('es', 'es')
df_cu_test = cargar_test('cu', 'cu')

df_test = pd.concat([df_mx_test, df_es_test, df_cu_test], ignore_index=True)
# Verificar duplicados en el test
print(f"Test: {len(df_test)} registros")
print(f"Distribución test IS_IRONIC: {df_test['IS_IRONIC'].value_counts().to_dict()}")

Test: 1800 registros
Distribución test IS_IRONIC: {0: 1201, 1: 599}


In [24]:
def preprocesar(texto):
    # Justificación: Ortega-Bueno et al. (2022) demuestran que mayúsculas
    # enfáticas, puntuación y emojis son señales estilísticas relevantes
    # para detección de ironía en español en redes sociales.
    # Plaza-del-Arco et al. (2022) aplican conversión de emojis a
    # descripciones textuales para preservar su carga semántica.

    # 1. Reemplazar URLs
    texto = re.sub(r'http\S+|www\S+', '[URL]', texto)
    # 2. Reemplazar menciones @usuario
    texto = re.sub(r'@\w+', '[USER]', texto)
    # 3. Convertir emojis a descripción textual en español (Plaza-del-Arco et al., 2022)
    texto = emoji.demojize(texto, language='es')
    # 4. Normalizar caracteres repetidos máx 2 (Ortega-Bueno et al., 2022)
    texto = re.sub(r'(.)\1{2,}', r'\1\1', texto)
    # 5. Conservar puntuación relevante como señal pragmática de ironía
    # ¿ ¡ ! ? se mantienen (Ortega-Bueno et al., 2022)
    texto = re.sub(r'[^\w\s\!\?\.\,\;\:\#\[\]áéíóúüñÁÉÍÓÚÜÑ¿¡]', ' ', texto)
    # 6. Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

In [25]:
# Aplicar preprocesamiento a train y test
df_train['MESSAGE_CLEAN'] = df_train['MESSAGE'].apply(preprocesar)
df_test['MESSAGE_CLEAN']  = df_test['MESSAGE'].apply(preprocesar)
print(f"Preprocesamiento aplicado — Train: {df_train.shape} | Test: {df_test.shape}")

Preprocesamiento aplicado — Train: (7197, 6) | Test: (1800, 6)


In [26]:
# Verificar resultado con un ejemplo por variante
for variante in ['mx', 'es', 'cu']:
    ejemplo = df_train[df_train['VARIANTE'] == variante].iloc[0]
    print(f"\n[{variante.upper()}]")
    print(f"ORIGINAL: {ejemplo['MESSAGE']}")
    print(f"LIMPIO:   {ejemplo['MESSAGE_CLEAN']}")


[MX]
ORIGINAL: Rica económicamente, pero muy pobre en objetividad.
LIMPIO:   Rica económicamente, pero muy pobre en objetividad.

[ES]
ORIGINAL: @ArmandoRuido007 @ANTI_MERMA50 @JoanTarda En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣
LIMPIO:   [USER] [USER] [USER] En vez de Joan Tarda van a llamarle No han tarda en callarle la boca :cara_revolviéndose_de_la_risa::cara_revolviéndose_de_la_risa:

[CU]
ORIGINAL: magnifico
LIMPIO:   magnifico


In [27]:
df_train.to_csv('../data/train_clean.csv', index=False)
df_test.to_csv('../data/test_clean.csv', index=False)
print("Guardado: train_clean.csv | test_clean.csv")

Guardado: train_clean.csv | test_clean.csv
